## Calculating a Typical Meteorological Year

This notebook walks through the process of calculating a [Typical Meteorological Year](https://nsrdb.nrel.gov/data-sets/tmy), an hourly dataset used for applications in energy and building systems modeling. Because this represents average rather than extreme conditions, an AMY dataset is not suited for designing systems to meet the worst-case conditions occurring at a location.

The TMY methodology here mirrors that of the Sandia/NREL TMY3 methodology, and uses historic and projected downscaled (bias corrected?) climate data available through the Cal-Adapt: Analytics Engine catalog. 

Please note, the Analytics Engine also has an *Average Meteorological Year* functionality. The methods shown throughout this notebook will soon replace the underlying backend `climakitae` code for the AMY in order to better address our user needs, i.e., we are working to replace the AMY with the TMY methods. We provide this walkthrough to demonstrate confidence in the "AMY to TMY" conversion process for our users. 

### Step 0: Set-up

Import the [climakitae](https://github.com/cal-adapt/climakitae) library and other dependencies.

In [ ]:
import climakitae as ck
import pandas as pd
import xarray as xr

from statsmodels.distributions.empirical_distribution import ECDF
import matplotlib.pyplot as plt

To use climakitae, load a new application:

In [ ]:
app = ck.Application()

### Step 1: Grab and process all required input data

The TMY3 method selects a "typical" month based on ten daily indices: max, min, and mean dry bulb (air) and dew point temperatures, the max and mean wind speed, global irradiance and direct irradiance (REFS here). Some of these indices are available in the Analytics Engine catalog, and some we will need to calculate ourselves.  

- 15-20 years is the recommended minimum, use 30? 
- Leap year data is included

#### Step 1a: Indices in catalog
- Max and min air temperature
- Max and mean wind speed
- Global irradiance

In [ ]:
# default selections applicable to all variables selected
app.selections.data_type = "Gridded"
app.selections.area_average = "Yes"
app.selections.scenario_historical = ["Historical Climate"]
app.selections.scenario_ssp = ["SSP 3-7.0 -- Business as Usual"] ##
app.selections.time_slice = (1990, 2020) # using 30 year as baseline to start
app.selections.resolution = "45 km"
app.selections.area_subset = "CA counties"
app.selections.cached_area = "Alameda County"

In [ ]:
# max air temp
app.selections.variable = "Daily maximum air temperature at 2m"
max_airtemp_data = app.retrieve()

In [ ]:
# min air temp
app.selections.variable = "Daily minimum air temperature at 2m"
min_airtemp_data = app.retrieve()

In [ ]:
# max wind speed
app.selections.variable = "Maximum wind speed at 10m"
max_windspd_data = app.retrieve()

In [ ]:
# mean wind speed
app.selections.variable = "Mean wind speed at 10m"
mean_windspd_data = app.retrieve()

In [ ]:
# global irradiance
app.selections.variable = "Instantaneous downwelling shortwave flux at bottom"
global_irradiance_data = app.retrieve()

#### Step 1b: Indices that need to be calculated
- Mean air temperature
- Max, min, and mean dew point temperature
- Direct irradiance (from diffuse and global)

In [ ]:
# switch to hourly timescale
app.selections.timescale = "hourly"

In [ ]:
# mean air temperature
app.selections.variable = "Air Temperature at 2m"
airtemp_data = app.retrieve()

mean_airtemp_data = airtemp_data.resample(time="1D").mean() # daily mean from hourly data
mean_airtemp_data.name = "Daily mean air temperature" # rename for clarity

In [ ]:
# dew point temperature
app.selections.variable = "Dew point temperature"
dewpt_data = app.retrieve()

max_dptemp_data = dewpt_data.resample(time="1D").max() # daily max dewpoint temp
max_dptemp_data.name = "Daily max dewpoint temperature" # rename for clarity

min_dptemp_data = dewpt_data.resample(time="1D").min() # daily min dewpoint temp
min_dptemp_data.name = "Daily min dewpoint temperature" # rename for clarity

mean_dptemp_data = dewpt_data.resample(time="1D").mean() # daily mean dewpoint temp
mean_dptemp_data.name = "Daily mean dewpoint temperature" # rename for clarity

In [ ]:
# direct irradiance
app.seletions.variable = "Shortwave surface downward diffuse irradiance"
diffuse_data = app.retrieve()

In [ ]:
# load all indices in
max_airtemp_data = app.load(max_airtemp_data.squeeze())
min_airtemp_data = app.load(min_airtemp_data.squeeze())
mean_airtemp_data = app.load(mean_airtemp_data.squeeze())
max_dptemp_data = app.load(max_dptemp_data.squeeze())
min_dptemp_data = app.load(min_dptemp_data.squeeze())
mean_dptemp_data = app.load(mean_dptemp_data.squeeze())
max_windspd_data = app.load(max_windspd_data.squeeze())
mean_windspd_data = app.load(mean_windspd_data.squeeze())
global_irradiance_data = app.load(global_irradiance_data.squeeze())
# direct_irradiance_data = app.load(direct_irradiance_data.squeeze())

In [ ]:
ds = xr.merge([max_airtemp_data, min_airtemp_data, mean_airtemp_data,
               max_dptemp_data, min_dptemp_data, mean_dptemp_data,
               max_windspd_data, mean_windspd_data,
               global_irradiance_data]) #, direct_irradiance_data])
ds

### Step 2: Calculate cumultative distribution functions

Notes on what a CDF is here: 

*** makes assumptions based on the distribution function here, fyi

#### Step 2a: Calculate long-term climatology CDFs for each index
For each month.....

In [ ]:
def plot_cdf(data):
    for i in range(4):
        cdf = ECDF(data.isel(simulation=i))
        _to_plot = plt.plot(cdf.x, cdf.y)
    return _to_plot

In [ ]:
plot_cdf(max_airtemp_data)

In [ ]:
# produce subplots that illustrates each climatological CDF with variable name provided, with month as a toggle selection

In [ ]:
max_airtemp_data

#### Step 2b: Calculate CDFs for each index for each year

Calculate CDF for each month and each variable: Gives proportion of values that are less than or equal to a specified value of index/variable
- Must exclude the following months: 
    - El Chichón: May 1982 until December 1984
    - Pinatubo: June 1991 to December 1994
    - Leap year days are excluded in the monthly CDFS

In [ ]:
def remove_volcanos(data):
    """Excludes specific months from CDF calculation due to volcanic eruptions"""
    elchichon = slice("1982-05-01", "1984-12-31")
    pinatubo = slice("1991-06-01", "1994-12-31")
    
    data_to_remove = data.sel(time=elchichon)
    data = data - data_to_remove
    
    data_to_remove = data.sel(time=pinatubo)
    data = data - data_to_remove
    
    return data


def remove_leapdays(data):
    """Drops Feb 29th"""
    
    return data

### Step 3: Compare climatology CDF to monthly CDF for each variable

#### Step 3a: Calculate the Finkelstein-Schafer statistic 
REF HERE
Absolute difference between long-term climatology CDF and candidate CDF divided by number of days per month

In [ ]:
def fs_statistic(cdf_climatology, cdf_month):
    """
    Calculates the Finkelstein-Schafer statistic:
    Absolute difference between long-term climatology and candidate CDF, divided by number of days in month
    """
    len_month = number_of_days_in_month
    da_fs = (cdf_climatology - cdf_month).abs().mean()
    return da_fs / len_month


# Absolute difference between long-term climatology CDF and candidate CDF divided by number of days in month
# Finkelstein-Schafer statistic (difference between long term  CDF and year CDF

#### Step 3b: Weight the F-S statistic
WS = Weight * F-S value

- Max/min air temp, max/min dry bulb temp, max/mean wind speed 1/20 each
- Mean air temp, mean dew point 2/20 each
- Global irradiance, direct irradiance 5/20 each

In [ ]:
def weighted_fs(da_fs):
    """Weights the F-S statistics based on TMY3 methodology"""
    
    
    
    return da_fs_weighted

#### Step 3c: Select candidate months for consideration
Once weighted, select 5 candidate months that have lowest weighted sums

In [ ]:
# for each month, rank all available years, lowest to highest
# select first five as "candidates"

#### Step 3d: Rank candidate months for selection
Rank with respect to closeness of month to long-term mean and median

"Typical" month selected by choosing among 5 months with lowest WS values the month with the smallest deviation from the long-term CDF

In [ ]:
# using 5 candidate months
# identify which month is closest to climatology

### Step 4: Consider persistence of weather patterns

#### Step 4a: Monthly mean and median, persistence of wx patterns
- Mean daily dry bulb temp
- Global horizontal radiation
- frequency and run length above 67th percentile/below 33rd percentile

Exclude month with longest run, month with most runs, month with zero runs

### Step 5: Concatenate months together

Months concatenated together and discontinuities at month interface are smoothed for 6 hours on either side using curve-fitting techniques

In [ ]:
# provide a list/table/output of which month comes from which year

### Step 6: Generate the TMY data outputs

Generally, the following is outputted using the TMY months:
- Date & time (UTC)
- T2m [°C] - Dry bulb (air) temperature.
- RH [%] - Relative Humidity.
- G(h) [W/m2] - Global horizontal irradiance.
- Gb(n) [W/m2] - Direct (beam) irradiance -- ***currently not available to calculate in AE -- but can be calculated here and added in as derived variable?***
- Gd(h) [W/m2] - Diffuse horizontal irradiance.
- IR(h) [W/m2] - Infrared radiation downwards -- ***Instantaneous downwelling longwave flux at bottom -- check that this is what we want here***
- WS10m [m/s] - Windspeed.
- WD10m [°] - Wind direction. -- ***currently not available to calculate in AE***
- SP [Pa] - Surface (air) pressure.

In [ ]:
# step 1 - using TMY selction of months
# step 2 - grab available variables at hourly timescales as dataarrays
# step 3 - concat variable dataarrays into pandas df
# step 4 - visualize (head) table
# step 5 - export table ==> this is the "TMY data file"